# Module 5: Evaluation

**Duration:** 30 minutes (plus 15 optional minutes for section 5.6)

How do you know if your RAG system is working? Feeling like the answers are good is not enough,
especially when you change something and want to know if it improved things.
In this module we measure the pipeline from Module 4 *before* we try to improve it in Module 6.

**By the end of this module you can:**
- explain why the retriever and the generator have to be evaluated separately
- use a gold dataset to compute Recall@k
- diagnose a retrieval miss by reading what was retrieved instead
- record a baseline that later changes can be compared against

---

## 5.1 What to evaluate and why

A RAG system has two components that can fail independently.
The **retriever** can fail by not returning the right chunks.
The **generator** can fail by ignoring the context or misreading it.
You need to evaluate both separately, not just look at the final answer:
a system that gets the right answer for the wrong reasons will fail on harder questions.

<img src="../images/rag_evaluation_triangle.svg" width="600" />

- **Faithfulness**: does the answer only contain claims supported by the retrieved context? (hallucination)
- **Relevance**: does the answer actually address what was asked? (off-topic or evasive answers)
- **Accuracy**: is the answer factually correct? (needs a reference answer)

Human judgement is slow and inconsistent, and it cannot tell you whether a change *improved*
things. Even a small, imperfect automated evaluation is more useful than intuition, because
it is *consistent* and *repeatable*.

In this module we focus on the retriever, because retrieval quality sets the upper bound for
everything that follows. Generation metrics are in the optional section 5.6.

## 5.2 The gold dataset

A gold dataset is a set of questions where you know the correct answer and where it can be
found. Ours is in `data/gold_dataset.json`. Each entry has:

- `question`: phrased the way a real user would ask it
- `expected_fragment`: a short, distinctive phrase that must appear in a chunk that answers the question
- `source`: the file the answer comes from

Most questions are about the fictional Nordlicht employee handbook. Notice that they are written
in everyday language ("work from Spain", "per diem", "cab") rather than with the words the
handbook uses. That is deliberate: real users do not know the vocabulary of your documents.

**What makes a good gold question?**
- Good: "What is the per diem?": specific, one definite answer in one place
- Bad: "What does the company do?": vague, the answer spans many chunks
- Bad: "Is the travel policy fair?": opinion, no factual answer

In [ ]:
import warnings

warnings.filterwarnings('ignore')

import json

from ragsst.ragtool import RAGTool

tool = RAGTool(data_path='../data/sample_docs', collection_name='workshop_docs')
tool.make_collection('../data/sample_docs', 'workshop_docs')  # adds missing files only

with open('../data/gold_dataset.json', encoding='utf-8') as f:
    gold_dataset = json.load(f)

print(f'Gold dataset: {len(gold_dataset)} questions\n')
for item in gold_dataset:
    print(f"{item['expected_fragment']:<38} {item['question']}")

### Why check the fragment and not just the file?

A simpler check is "did a chunk from the right *file* come back?". With only five files that
is almost always true, so the score sits at 1.0 and cannot show any improvement. Checking that
a retrieved chunk *contains the answer* is stricter and much more informative.

> **Small is fine for a workshop.** Twelve questions are enough to see patterns, but not to
> prove that one technique is better than another. In production you want 50 to 100 questions.

## 5.3 Recall@k

Recall@k answers: *for what share of the questions does a chunk with the answer appear in the
top k retrieved chunks?* It measures the retriever alone, needs no LLM call, and is fast.

How to read it:
- **Recall@1 = 1.0**: the right chunk is always the top result.
- **Recall@3 = 1.0, Recall@1 = 0.6**: retrieval finds the right chunk but ranks it too low.
  Re-ranking is a good candidate.
- **Recall@5 = 0.7**: 30% of the time the answer is not even in the top 5. That is a chunking,
  embedding model or data problem.

In [ ]:
def baseline_retrieve(query: str, k: int) -> list[str]:
    """Plain vector search: the top-k chunks for the query."""
    return tool.collection.query(query_texts=[query], n_results=k)['documents'][0]


def is_hit(chunks: list[str], expected_fragment: str) -> bool:
    return any(expected_fragment.lower() in chunk.lower() for chunk in chunks)


def recall_at_k(retrieve, dataset: list[dict], k: int) -> float:
    hits = [is_hit(retrieve(item['question'], k), item['expected_fragment']) for item in dataset]
    return sum(hits) / len(hits)


print(f"{'k':>4}  {'Recall@k':>10}")
print('-' * 18)
for k in [1, 2, 3, 5]:
    print(f'{k:>4}  {recall_at_k(baseline_retrieve, gold_dataset, k):>10.2f}')

Notice that `recall_at_k` takes the *retrieval function* as an argument. In Module 6 we write
one retrieval function per technique and score all of them with exactly the same code.
The same helpers are available as `ragsst.evaluation`, so later notebooks can import them.

## 5.4 Diagnose the misses

A score tells you *that* something is wrong, not *what*. The cell below shows, for every
question the baseline misses at k=1, which chunk came back instead.

**Think, pair, share (8 minutes):** take one minute alone to read the misses. For each, decide
why the wrong chunk won. Then compare with your neighbour. Typical causes:
- an outdated document that looks almost the same as the current one
- the user's words differ from the document's words
- the right section is split across chunks or mixed with another topic

Which of the misses could a better *ranking* fix? Which would need better *documents*?

In [ ]:
K = 1
for item in gold_dataset:
    chunks = baseline_retrieve(item['question'], K)
    if is_hit(chunks, item['expected_fragment']):
        print(f"OK    {item['question']}")
    else:
        top = chunks[0].replace('\n', ' ')
        print(f"MISS  {item['question']}")
        print(f"        expected: '{item['expected_fragment']}'")
        print(f"        got:      {top[:180]}...")

## 5.5 Write down your baseline

| Pipeline | Recall@1 | Recall@3 |
|---|---|---|
| Baseline (Module 4) | | |

Copy your numbers into the table. Module 6 starts by recomputing them, so you can check that
nothing changed in between.

From now on, every change follows the same loop:

1. **Baseline**: measure the current pipeline.
2. **Hypothesis**: "re-ranking will improve Recall@1, because the right chunk is often in the top 3".
3. **Change**: implement it.
4. **Measure**: same gold set, same metric.
5. **Decide**: did it improve, by how much, and is the extra latency or cost worth it?

Even "obvious" improvements sometimes make things worse. You will see an example in Module 6.

## 5.6 Optional: answer accuracy and LLM-as-judge

Recall@k says nothing about the generated answer. Two cheap checks:

- **Answer accuracy**: does the generated answer contain the expected fragment?
- **Faithfulness**: we ask the LLM itself whether the answer contains claims that are not in the
  context ("LLM-as-judge"). Fast, but the judge can also be wrong, especially a small one.

> On a laptop without a GPU each LLM call takes 10 to 30 seconds. The cell below needs two calls
> per question, so it only runs on the first four questions.

In [ ]:
from os import getenv
from urllib.parse import urljoin

import requests

OLLAMA_URL = urljoin(getenv('OLLAMA_HOST', 'http://localhost:11434'), 'api')
MODEL = 'llama3.2:3b'


def generate(prompt: str, temp: float = 0.3) -> str:
    r = requests.post(
        OLLAMA_URL + '/generate',
        json={'model': MODEL, 'prompt': prompt, 'stream': False,
              'options': {'temperature': temp}}
    )
    return json.loads(r.text).get('response', '')


def check_faithfulness(question: str, context: str, answer: str) -> str:
    prompt = (
        'Given the context and the answer below, decide if the answer contains any '
        'information that is NOT in the context. '
        'Reply with FAITHFUL if everything in the answer is supported by the context, '
        'or HALLUCINATION followed by a brief explanation if not.\n\n'
        f'Context:\n{context}\n\n'
        f'Question: {question}\n'
        f'Answer: {answer}\n'
        'Verdict:'
    )
    return generate(prompt, temp=0.1)


for item in gold_dataset[:4]:
    context = '\n\n'.join(baseline_retrieve(item['question'], 3))
    answer = generate(tool.get_context_prompt(item['question'], context))
    correct = item['expected_fragment'].lower() in answer.lower()
    verdict = check_faithfulness(item['question'], context, answer)
    print(f"Q: {item['question']}")
    print(f'A: {answer[:200]}')
    print(f"Contains '{item['expected_fragment']}': {correct}")
    print(f'Judge: {verdict.strip()[:200]}')
    print()

## 5.7 Outlook: RAGAS

[RAGAS](https://docs.ragas.io/) is an open-source framework that automates RAG evaluation with
standard metrics:

| Metric | What it measures | Requires |
|--------|-----------------|---------|
| **Faithfulness** | Are all claims in the answer supported by the context? | Answer + context |
| **Answer Relevancy** | Does the answer address the question asked? | Answer + question |
| **Context Precision** | Are the retrieved chunks relevant, or is there noise? | Context + question |
| **Context Recall** | Did the retriever find everything needed to answer? | Context + reference answer |

For faithfulness it asks an LLM to split the answer into single claims, checks each claim
against the context, and reports the share of supported claims. That is much more robust than
our fragment check.

We do not run RAGAS in the workshop: it needs extra packages and many LLM calls, which is slow
with a local model. A minimal setup looks like this (by default it uses an OpenAI model; it can
be configured to use a local model through Ollama instead):

```python
# uv add ragas datasets
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import answer_relevancy, faithfulness

data = {
    'question': questions,       # list[str]
    'answer': answers,           # generated answers
    'contexts': contexts,        # list[list[str]], retrieved chunks per question
    'ground_truth': references,  # reference answers
}
print(evaluate(Dataset.from_dict(data), metrics=[faithfulness, answer_relevancy]))
```

Even without running it, the RAGAS metric names give you a shared vocabulary for discussing RAG
quality with your team.

---

**Exercises**

1. Add three questions about the Sherlock Holmes PDF to `gold_dataset` (pick fragments that
   really appear in the text). Does Recall@3 change?

2. Compute Recall@k for k from 1 to 10. Where does the curve flatten? What does it cost to give
   the LLM more chunks (think of section 4.3 and "lost in the middle")?

3. Some misses are caused by the superseded 2023 travel policy. Name two ways to fix this
   *without* touching the retriever. What does that tell you about data quality in RAG projects?

---

**Further reading**

- RAGAS paper: https://arxiv.org/abs/2309.15217
- RAGAS docs: https://docs.ragas.io/
- DeepEval RAG evaluation: https://deepeval.com/guides/guides-rag-evaluation
- BERGEN benchmarking library: https://arxiv.org/abs/2407.01102